1. Named Entity Recognition lab. A CoNLL-style file ner_data.txt contains one token and its BIO tag
(B-PER, I-PER, B-LOC, I-LOC, B-ORG, I-ORG, O) per line, separated by a space, with a blank
line between sentences. Implement NER in Python using two methods and compare them.
[10][CO04][BTL3]

a. Read the file into a list of sentences of (token, tag) pairs and split it into 80% training and 20%
test sentences. [2]

b. Method 1: use a pre-trained spaCy model to predict the entities of the test sentences and convert
them to BIO tags (map PERSON → PER, GPE/LOC → LOC, ORG → ORG). [2]

c. Method 2: define a token feature function (word, lower-case word, suffix, capitalisation, digit
check, previous and next word) and train a CRF on the training sentences. [3]

d. Evaluate both methods on the test set by printing the precision, recall and F1-score for each
entity type, and state which method performs better. [3]


In [2]:
import spacy
from spacy.pipeline import EntityRuler
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

with open ("ner.txt", "r", encoding="utf-8") as f:
    for l in f:
        print(l)

I O

want O

to O

buy O

a O

Sony B-ORG

PlayStation B-PROD

5 I-PROD

Pro I-PROD

online O

. O



The O

retailer O

Best B-ORG

Buy I-ORG

is O

selling O

it O

for O

699.99 B-MONEY

USD I-MONEY

. O



Shipping O

to O

New B-LOC

York I-LOC

takes O

three B-CARDINAL

days O

. O



In [ ]:
import random
import spacy

nlp = spacy.load("en_core_web_sm")

def load_data_from_raw_text(text_data):
    """
    Parses a tab-separated or space-separated CoNLL-style NER string into a list of sentences.
    Each sentence contains a list of (token, tag) tuples.
    """
    sentences = []
    current_sentence = []
    
    for line in text_data.strip().splitlines():
        line = line.strip()
        if not line:
            if current_sentence:
                sentences.append(current_sentence)
                current_sentence = []
        else:
            parts = line.split()
            if len(parts) >= 2:
                token, tag = parts[0], parts[-1]
                current_sentence.append((token, tag))
                
    if current_sentence:
        sentences.append(current_sentence)
        
    return sentences


def filter_non_important_words(sentences, stop_words):
    """
    Filters out specified non-important words from the dataset while keeping non-O tags intact.
    """
    filtered_sentences = []
    stop_words_set = set(word.lower() for word in stop_words)
    
    for sent in sentences:
        new_sent = []
        for token, tag in sent:
            # We retain words unless they are explicitly marked non-important and tagged as 'O'
            if token.lower() in stop_words_set and tag == "O":
                continue
            new_sent.append((token, tag))
            
        if new_sent:
            filtered_sentences.append(new_sent)
            
    return filtered_sentences


def display_corpus(sentences, num_preview=3):
    """
    Prints a clean human-readable preview of the tokenized corpus.
    """
    print("\n--- Processed Corpus Preview ---")
    for i, sent in enumerate(sentences[:num_preview]):
        formatted_sent = " ".join([f"{token}/{tag}" for token, tag in sent])
        print(f"Sentence {i+1}: {formatted_sent}")
    print("---------------------------------\n")


def custom_entity_lookup(text, custom_entity_map):
    """
    Checks user input string against custom-defined entity keywords.
    """
    found_entities = []
    words = text.split()
    
    for word in words:
        clean_word = word.strip(".,!?").lower()
        if clean_word in custom_entity_map:
            found_entities.append((word, custom_entity_map[clean_word]))
            
    return found_entities



# 1. Allow user to enter the corpus interactively
print("Enter your CoNLL-formatted corpus (Token Tag per line). Press Enter twice on an empty line when finished:")
user_lines = []
while True:
    try:
        line = input()
        if not line and (len(user_lines) > 0 and user_lines[-1] == ""):
            break
        user_lines.append(line)
    except EOFError:
        break

raw_corpus_input = "\n".join(user_lines)

if not raw_corpus_input.strip():
    print("\n[INFO] No corpus entered. Using built-in default dataset.")
    raw_corpus_input = """
Apple B-ORG
is O
looking O
at O
buying O
U.K. B-LOC
startup O
for O
$1 O
billion O

Tim B-PER
Cook I-PER
is O
the O
CEO O
of O
Apple B-ORG
in O
California B-LOC
"""

all_sentences = load_data_from_raw_text(raw_corpus_input)

# 2. Allow user to specify non-important words
user_stopwords = input("\nEnter non-important words to filter out (comma-separated, e.g., 'is, at, for'): ")
stop_words_list = [w.strip() for w in user_stopwords.split(",") if w.strip()]

# 3. Regenerate and display the corpus
cleaned_sentences = filter_non_important_words(all_sentences, stop_words_list)
display_corpus(cleaned_sentences)

# 4. Allow user to create custom entities and test them
custom_entities = {}
print("Add custom entities to the system (e.g., word -> entity type).")
while True:
    entity_input = input("Enter word and entity tag separated by space (or press Enter to finish): ").strip()
    if not entity_input:
        break
    parts = entity_input.split()
    if len(parts) == 2:
        word, ent_tag = parts[0].lower(), parts[1].upper()
        custom_entities[word] = ent_tag
        print(f"Added: {word} -> {ent_tag}")
    else:
        print("Invalid format. Please use: <word> <TAG>")

if custom_entities:
    test_phrase = input("\nEnter a sentence to check for your custom entities: ")
    detected = custom_entity_lookup(test_phrase, custom_entities)
    print("\nCustom Entity Detection Results:")
    if detected:
        for word, tag in detected:
            print(f" - Word: '{word}' | Entity: {tag}")
    else:
        print("No custom entities detected in the input.")


# Model Evaluation Pipeline (spaCy & CRF)

random.seed(42)
shuffled_sentences = cleaned_sentences.copy()
random.shuffle(shuffled_sentences)

split_idx = int(len(shuffled_sentences) * 0.8)
train_sents = shuffled_sentences[:split_idx]
test_sents = shuffled_sentences[split_idx:]

if len(test_sents) == 0:
    train_sents, test_sents = cleaned_sentences, cleaned_sentences

print(f"\nDataset Statistics -> Total: {len(cleaned_sentences)} | Train: {len(train_sents)} | Test: {len(test_sents)}")


# Method 1: spaCy Predictor
def predict_spacy(sentence):
    tokens = [tok for tok, tag in sentence]
    doc = spacy.tokens.Doc(nlp.vocab, words=tokens)
    nlp.get_pipe("ner")(doc)
    
    predicted_tags = ["O"] * len(tokens)
    tag_map = {"PERSON": "PER", "GPE": "LOC", "LOC": "LOC", "ORG": "ORG"}
    
    for ent in doc.ents:
        mapped_label = tag_map.get(ent.label_, None)
        if mapped_label:
            for i in range(ent.start, ent.end):
                prefix = "B-" if i == ent.start else "I-"
                predicted_tags[i] = f"{prefix}{mapped_label}"
                
    return predicted_tags


# Method 2: Custom CRF Feature Extraction
def token2features(sent, i):
    word = sent[i][0]
    features = {
        'bias': 1.0,
        'word': word,
        'lower': word.lower(),
        'suffix_2': word[-2:],
        'suffix_3': word[-3:],
        'is_capitalized': word[0].isupper(),
        'is_all_caps': word.isupper(),
        'is_digit': word.isdigit(),
    }
    
    if i > 0:
        prev_word = sent[i-1][0]
        features.update({
            '-1:word': prev_word,
            '-1:lower': prev_word.lower(),
            '-1:is_capitalized': prev_word[0].isupper(),
        })
    else:
        features['BOS'] = True

    if i < len(sent) - 1:
        next_word = sent[i+1][0]
        features.update({
            '+1:word': next_word,
            '+1:lower': next_word.lower(),
            '+1:is_capitalized': next_word[0].isupper(),
        })
    else:
        features['EOS'] = True

    return features

def sent2features(sent):
    return [token2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]
X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

# CRF Model Training
try:
    from sklearn_crfsuite import CRF
    crf = CRF(algorithm='lbfgs', c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True)
    crf.fit(X_train, y_train)
    crf_preds = crf.predict(X_test)
except ImportError:
    import pycrfsuite
    trainer = pycrfsuite.Trainer(verbose=False)
    for xseq, yseq in zip(X_train, y_train):
        trainer.append(xseq, yseq)
    trainer.set_params({'c1': 0.1, 'c2': 0.1, 'max_iterations': 100})
    trainer.train('ner_model.crfsuite')
    
    tagger = pycrfsuite.Tagger()
    tagger.open('ner_model.crfsuite')
    crf_preds = [tagger.tag(xseq) for xseq in X_test]

# Model Evaluation
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

spacy_preds = [predict_spacy(s) for s in test_sents]

if hasattr(crf_preds, "tolist"):
    crf_preds = crf_preds.tolist()
crf_preds = [list(sent) for sent in crf_preds]

print("\n==================== spaCy Baseline Evaluation ====================")
print(classification_report(y_test, spacy_preds))

print("\n==================== Custom CRF Model Evaluation ====================")
print(classification_report(y_test, crf_preds))

# Summary Comparison
spacy_p = precision_score(y_test, spacy_preds)
spacy_r = recall_score(y_test, spacy_preds)
spacy_f1 = f1_score(y_test, spacy_preds)

crf_p = precision_score(y_test, crf_preds)
crf_r = recall_score(y_test, crf_preds)
crf_f1 = f1_score(y_test, crf_preds)

print("=" * 60)
print("FINAL MODEL COMPARISON:")
print(f"spaCy Model -> Precision: {spacy_p:.4f} | Recall: {spacy_r:.4f} | F1: {spacy_f1:.4f}")
print(f"CRF Model   -> Precision: {crf_p:.4f} | Recall: {crf_r:.4f} | F1: {crf_f1:.4f}")
print("=" * 60)

Enter your CoNLL-formatted corpus (Token Tag per line). Press Enter twice on an empty line when finished:

[INFO] No corpus entered. Using built-in default dataset.

--- Processed Corpus Preview ---
Sentence 1: Apple/B-ORG is/O looking/O at/O buying/O U.K./B-LOC startup/O for/O $1/O billion/O
Sentence 2: Tim/B-PER Cook/I-PER is/O the/O CEO/O of/O Apple/B-ORG in/O California/B-LOC
---------------------------------

Add custom entities to the system (e.g., word -> entity type).

Dataset Statistics -> Total: 2 | Train: 1 | Test: 1

==================== spaCy Baseline Evaluation ====================
              precision    recall  f1-score   support

         LOC       1.00      1.00      1.00         1
         ORG       1.00      1.00      1.00         1

   micro avg       1.00      1.00      1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2


==================== Custom CRF Model Evaluation ====================

d:\Studies\3rd year\Semestser 5\Natural Language Processing\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


2. Dependency parsing lab. Consider the sentences S1: “The manager approved the new budget.” and
S2: “Students submitted assignments and the teacher graded them.” Use spaCy’s en_core_web_sm
model.
[10][CO05][BTL4]

a. Parse both sentences and print a table with the columns token, POS tag, dependency label and
head word. [3]

b. Visualise the dependency tree of S1 using displaCy and, from the parse, identify the root, the
subject and the object. [2]

c. Write a function extract_svo(doc) that returns (subject, verb lemma, object) triples, and test it
on S1 and S2. [3]

d. The gold and predicted parses of “The quick fox jumps over the lazy dog” are given below as
head index (0 = root) and label. Write a function to compute the Unlabeled Attachment Score
(UAS) and the Labeled Attachment Score (LAS), and report their values. [2


In [ ]:
import pandas as pd
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")


def filter_unwanted_words(sentence_text, stop_words):
    """
    Removes user-specified non-important words from a given sentence.
    """
    words = sentence_text.split()
    stop_words_set = set(w.lower() for w in stop_words)
    cleaned_words = [w for w in words if w.lower() not in stop_words_set]
    return " ".join(cleaned_words)


def extract_svo(doc):
    """
    Extracts Subject-Verb-Object (SVO) triplets from a parsed spaCy Doc object.
    """
    svos = []
    for token in doc:
        if token.dep_ == "nsubj":
            subject = token
            verb = token.head
            obj = None
            for child in verb.children:
                if child.dep_ in ("dobj", "attr", "prep", "pobj"):
                    obj = child
                    break
            if obj:
                svos.append((subject.text, verb.text, obj.text))
    return svos


def check_custom_entity_role(doc, target_word, custom_label):
    """
    Searches for a user-created entity in the parsed tree and identifies
    whether it acts as a Subject, Object, or general token.
    """
    matches = []
    for token in doc:
        if token.text.lower() == target_word.lower():
            role = "Other Dependency"
            if "subj" in token.dep_:
                role = "Subject"
            elif "obj" in token.dep_ or token.dep_ == "attr":
                role = "Object"
            elif token.pos_ == "VERB":
                role = "Predicate/Verb"

            matches.append(
                {
                    "Word": token.text,
                    "Custom Label": custom_label,
                    "Syntactic Role": role,
                    "Governing Head": token.head.text,
                }
            )
    return matches


# 1. Interactive Inputs from End User
print("==================================================")
print("     DEPENDENCY PARSER & SVO TRIPLE EXTRACTOR     ")
print("==================================================")

# Allow user to enter their own corpus/sentences
print(
    "\n1. Enter your sentences (separate multiple sentences with '|' or press"
    " Enter for defaults):"
)
user_corpus_input = input("> ").strip()

if user_corpus_input:
    sentences = [s.strip() for s in user_corpus_input.split("|") if s.strip()]
else:
    print("-> No input provided. Using default sentences.")
    sentences = [
        "The manager approved the new budget.",
        "Students submitted assignments and the teacher graded them.",
    ]

# Allow user to specify non-important words
print("\n2. Enter non-important words to remove (comma-separated, e.g., 'the, a'):")
user_stopwords_input = input("> ").strip()
stopwords = [w.strip() for w in user_stopwords_input.split(",") if w.strip()]

# 3. Regenerate the corpus and present the filtered sentences
print("\n--- 3. REGENERATED CORPUS (After Filtering Non-Important Words) ---")
regenerated_sentences = []
for idx, sent_text in enumerate(sentences, 1):
    clean_text = filter_unwanted_words(sent_text, stopwords)
    regenerated_sentences.append(clean_text)
    print(f"  Sentence {idx} (Original)   : {sent_text}")
    print(f"  Sentence {idx} (Regenerated): {clean_text}\n")

# 4. Allow user to create a custom entity and check for it
print("4. Custom Entity Creation & Syntactic Verification:")
custom_word = input(
    "Enter a specific word to track as a custom entity (e.g., 'manager' or"
    " 'assignments'): "
).strip()
custom_label = (
    input(f"Enter a custom label for '{custom_word}' (e.g., PERSON / ASSET): ")
    .strip()
    .upper()
)


# Processing & Evaluation Pipelines
def parse_and_analyze(sentence_text):
    print("\n" + "=" * 65)
    print(f"ANALYSIS FOR: '{sentence_text}'")
    print("=" * 65)

    doc = nlp(sentence_text)

    # Display Parse Table
    print("\n--- Dependency Parse Table ---")
    print(
        f"{'Token':<12} | {'POS Tag':<8} | {'Dependency Label':<16} |"
        f" {'Head Token':<12} | {'Children'}"
    )
    print("-" * 65)

    for token in doc:
        children = [child.text for child in token.children]
        print(
            f"{token.text:<12} | {token.pos_:<8} | {token.dep_:<16} |"
            f" {token.head.text:<12} | {', '.join(children)}"
        )

    # Visualizer output call
    try:
        displacy.render(doc, style="dep", jupyter=True)
    except Exception:
        print("Error rendering dependency parse.")

    # Extract SVO Triplets
    print("\n--- Extracted SVO Triplets ---")
    svos = extract_svo(doc)
    if svos:
        for subj, verb, obj in svos:
            print(f"  • Subject: '{subj}' ---> Verb: '{verb}' ---> Object: '{obj}'")
    else:
        print("  • No complete Subject-Verb-Object triplets found.")

    # Check for User-Created Custom Entity
    if custom_word and custom_label:
        entity_matches = check_custom_entity_role(doc, custom_word, custom_label)
        print(f"\n--- Custom Entity Check ({custom_label}) ---")
        if entity_matches:
            for match in entity_matches:
                print(
                    f"  • Found '{match['Word']}' labeled as [{match['Custom Label']}]"
                )
                print(
                    f"    Syntactic Role: {match['Syntactic Role']} | Attached"
                    f" Head: {match['Governing Head']}"
                )
        else:
            print(
                f"  • Word '{custom_word}' was not found in this sentence."
            )


# Run parse analysis on regenerated sentences
for sent in regenerated_sentences:
    parse_and_analyze(sent)


# Part D (UAS & LAS)

gold_parse = [
    {"token": "The", "head": 3, "label": "det"},
    {"token": "quick", "head": 3, "label": "amod"},
    {"token": "fox", "head": 4, "label": "nsubj"},
    {"token": "jumps", "head": 0, "label": "ROOT"},
    {"token": "over", "head": 4, "label": "prep"},
    {"token": "the", "head": 8, "label": "det"},
    {"token": "lazy", "head": 8, "label": "amod"},
    {"token": "dog", "head": 5, "label": "pobj"},
]

pred_parse = [
    {"token": "The", "head": 3, "label": "det"},
    {"token": "quick", "head": 3, "label": "amod"},
    {"token": "fox", "head": 4, "label": "nsubj"},
    {"token": "jumps", "head": 0, "label": "ROOT"},
    {"token": "over", "head": 4, "label": "prep"},
    {"token": "the", "head": 8, "label": "det"},
    {"token": "lazy", "head": 8, "label": "amod"},
    {"token": "dog", "head": 4, "label": "dobj"},
]


def compute_attachment_scores(gold, pred):
    """
    Calculates Unlabeled Attachment Score (UAS) and Labeled Attachment Score (LAS).
    """
    correct_unlabeled = 0
    correct_labeled = 0
    total_tokens = len(gold)

    for g, p in zip(gold, pred):
        head_match = g["head"] == p["head"]
        label_match = g["label"] == p["label"]

        if head_match:
            correct_unlabeled += 1
            if label_match:
                correct_labeled += 1

    uas = correct_unlabeled / total_tokens
    las = correct_labeled / total_tokens

    return uas, las


uas_score, las_score = compute_attachment_scores(gold_parse, pred_parse)

print("\n" + "=" * 65)
print("PARSER EVALUATION METRICS (Part D)")
print("=" * 65)
print(f"Total Evaluated Tokens : {len(gold_parse)}")
print(f"Unlabeled Attachment Score (UAS) : {uas_score:.4f} ({uas_score * 100:.2f}%)")
print(f"Labeled Attachment Score (LAS)   : {las_score:.4f} ({las_score * 100:.2f}%)")
print("=" * 65)

     DEPENDENCY PARSER & SVO TRIPLE EXTRACTOR     

1. Enter your sentences (separate multiple sentences with '|' or press Enter for defaults):
-> No input provided. Using default sentences.

2. Enter non-important words to remove (comma-separated, e.g., 'the, a'):

--- 3. REGENERATED CORPUS (After Filtering Non-Important Words) ---
  Sentence 1 (Original)   : The manager approved the new budget.
  Sentence 1 (Regenerated): The manager approved the new budget.

  Sentence 2 (Original)   : Students submitted assignments and the teacher graded them.
  Sentence 2 (Regenerated): Students submitted assignments and the teacher graded them.

4. Custom Entity Creation & Syntactic Verification:

ANALYSIS FOR: 'The manager approved the new budget.'

--- Dependency Parse Table ---
Token        | POS Tag  | Dependency Label | Head Token   | Children
-----------------------------------------------------------------
The          | DET      | det              | manager      | 
manager      | NOUN    


--- Extracted SVO Triplets ---
  • Subject: 'manager' ---> Verb: 'approved' ---> Object: 'budget'

ANALYSIS FOR: 'Students submitted assignments and the teacher graded them.'

--- Dependency Parse Table ---
Token        | POS Tag  | Dependency Label | Head Token   | Children
-----------------------------------------------------------------
Students     | NOUN     | nsubj            | submitted    | 
submitted    | VERB     | ROOT             | submitted    | Students, assignments, and, graded
assignments  | NOUN     | dobj             | submitted    | 
and          | CCONJ    | cc               | submitted    | 
the          | DET      | det              | teacher      | 
teacher      | NOUN     | nsubj            | graded       | the
graded       | VERB     | conj             | submitted    | teacher, them, .
them         | PRON     | dobj             | graded       | 
.            | PUNCT    | punct            | graded       | 



--- Extracted SVO Triplets ---
  • Subject: 'Students' ---> Verb: 'submitted' ---> Object: 'assignments'
  • Subject: 'teacher' ---> Verb: 'graded' ---> Object: 'them'

PARSER EVALUATION METRICS (Part D)
Total Evaluated Tokens : 8
Unlabeled Attachment Score (UAS) : 0.8750 (87.50%)
Labeled Attachment Score (LAS)   : 0.8750 (87.50%)
